# Pancake Stayman

**Research question:** When Opener bids 1NT and Responder holds a flat hand with a matching 4-card major, does playing in the major suit game gain or lose compared to 3NT?

"Pancake" hands are the flattest possible — 4333 distributions. These are the hands where Stayman is *least* expected to gain, because:
- The 4–4 major suit fit offers little ruffing value when both hands are completely flat.
- Declarer in a major suit contract must lose a trump trick on any 4–2 break, whereas 3NT has no such vulnerability.

We quantify the effect using a double-dummy simulation.

## Setup

- **South** is a 1NT opener: balanced 15–17 HCP (`any 4333 + any 4432 + any 5332`).
- **North** is a game-forcing responder with no slam interest: 4333 or 3433 shape, 9–14 HCP.
  - `4333`: North has exactly 4 spades (will bid Stayman looking for a spade fit).
  - `3433`: North has exactly 4 hearts (will bid Stayman looking for a heart fit).

### Auction outcomes

We restrict to deals where Stayman *always* finds a fit, so every deal is a direct comparison between 3NT and the major suit game.

| Strategy | Contract |
|---|---|
| **Raise** | 3NT–S always |
| **Stayman** | 4♥–S if both hold 4+ hearts; 4♠–S if both hold 4+ spades |

In [ ]:
import bridgepandas as bp
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

## 1. Generate deals

North's shape and HCP are expressed as a `HandSet` so BDD sampling handles the heavy lifting; `accept` only checks South's shape and the major-suit fit. This is roughly 6× faster than pure accept/reject at N=10,000.

In [ ]:
_SOUTH_SHAPES = {(5, 3, 3, 2), (4, 4, 3, 2), (4, 3, 3, 3)}  # any 5332 / 4432 / 4333

# North is exactly 4333 or 3433 (D=3, C=3 fixed; one major is 4, the other 3)
north_hs = (
    (bp.h.HCP >= 9) & (bp.h.HCP <= 14)
    & (bp.h.DIAMONDS == 3) & (bp.h.CLUBS == 3)
    & ((bp.h.SPADES == 4) | (bp.h.HEARTS == 4))
)
south_hs = (bp.h.HCP >= 15) & (bp.h.HCP <= 17)

def accept(deal):
    s, n = deal.south, deal.north
    return (
        s.shape in _SOUTH_SHAPES
        and ((n.spades == 4 and s.spades >= 4) or (n.hearts == 4 and s.hearts >= 4))
    )

N = 10_000
df = bp.random_deals(N, south=south_hs, north=north_hs, accept=accept, seed=42)
print(f"{len(df)} deals generated")

## 2. Verify constraints

In [ ]:
pd.DataFrame({
    'south_hcp':    df['south'].hcp,
    'south_spades': df['south'].spades,
    'south_hearts': df['south'].hearts,
    'north_hcp':    df['north'].hcp,
    'north_spades': df['north'].spades,
    'north_hearts': df['north'].hearts,
}).describe().round(2)

In [ ]:
south_shapes = pd.Series([df.iloc[i].south.shape for i in range(N)])
print("South shapes:")
print(south_shapes.value_counts().to_string())
print()
north_has_4s = df['north'].spades == 4
print(f"North 4333 (4 spades): {north_has_4s.sum()}  ({north_has_4s.mean():.1%})")
print(f"North 3433 (4 hearts): {(~north_has_4s).sum()}  ({(~north_has_4s).mean():.1%})")

## 3. Determine Stayman contracts

North always has exactly one 4-card major. Stayman finds a fit if and only if South also holds 4+ cards in that suit.

In [ ]:
heart_fit = (df['north'].hearts >= 4) & (df['south'].hearts >= 4)
spade_fit = (df['north'].spades >= 4) & (df['south'].spades >= 4)

stayman_contracts = pd.Series(
    np.where(heart_fit, '4H-S', '4S-S')
)

print("Stayman auction outcomes:")
print(f"  4\u2665\u2013S (heart fit): {heart_fit.sum():4d}  ({heart_fit.mean():.1%})")
print(f"  4\u2660\u2013S (spade fit): {spade_fit.sum():4d}  ({spade_fit.mean():.1%})")

## 4. Double-dummy scoring

We use neither-vulnerable throughout, but we could change to NS vulnerable with `VULN = 'N'`.

In [ ]:
VULN = '-'  # neither vulnerable

bp.add_dds(df, '3N-S',           'score_3nt',     VULN)
bp.add_dds(df, stayman_contracts, 'score_stayman', VULN)

df[['score_3nt', 'score_stayman']].head(10)

## 5. IMP differential

`imp_diff > 0` means 3NT scored better on that board; `imp_diff < 0` means Stayman scored better.

In [ ]:
df['imp_diff'] = (df['score_3nt'] - df['score_stayman']).map(bp.scorediff_imps)
df['outcome']  = np.where(heart_fit, '4H fit', '4S fit')

df[['score_3nt', 'score_stayman', 'imp_diff', 'outcome']].head(10)

## 6. Statistical analysis

In [ ]:
mean_diff       = df['imp_diff'].mean()
std_diff        = df['imp_diff'].std()
se              = std_diff / np.sqrt(N)
t_stat, p_value = stats.ttest_1samp(df['imp_diff'], 0)
ci_lo, ci_hi    = mean_diff - 1.96 * se, mean_diff + 1.96 * se

print("IMP differential  (positive = 3NT better, negative = Stayman better)")
print(f"  n         = {N}")
print(f"  mean      = {mean_diff:+.3f} IMPs")
print(f"  std dev   = {std_diff:.3f}")
print(f"  std error = {se:.3f}")
print(f"  95% CI    = ({ci_lo:+.3f}, {ci_hi:+.3f})")
print(f"  t         = {t_stat:.3f}")
print(f"  p-value   = {p_value:.4f}  (two-sided, H\u2080: mean = 0)")
print()
if p_value < 0.05:
    winner = '3NT' if mean_diff > 0 else 'Stayman'
    print(f"\u2192 {winner} is significantly better (p < 0.05).")
else:
    print(f"\u2192 No significant difference detected (p = {p_value:.3f}).")

## 7. Does Responder's strength matter?

With fewer HCP, the combined strength is closer to the game threshold. A suit contract needs 10 tricks vs. 9 for 3NT, but on borderline hands the extra ruffing potential (even with flat shape) might tip the balance. We repeat the analysis at each possible North HCP count.

In [ ]:
hcp_breakdown = (
    df.groupby(df['north'].hcp)['imp_diff']
    .agg(count='count', mean='mean', std='std')
    .assign(se=lambda d: d['std'] / np.sqrt(d['count']),
            ci_lo=lambda d: d['mean'] - 1.96 * d['se'],
            ci_hi=lambda d: d['mean'] + 1.96 * d['se'])
)
print("IMP differential by North HCP  (positive = 3NT better)")
print(hcp_breakdown[['count', 'mean', 'se', 'ci_lo', 'ci_hi']].round(3).to_string())